In [1]:
# ============================================================
# 1. Configuration & Imports
# ============================================================
import os
import glob
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import open3d as o3d
import laspy
from tqdm.auto import tqdm
import pandas as pd

from torch_cluster import knn as tc_knn
from torch_scatter import scatter_softmax, scatter_add, scatter_max, scatter_mean

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version :", torch.__version__)
print("Running on      :", DEVICE)

CONFIG = {
    # ---- data folders ----
    "train_dir": "data/train",          
    "test_dir":  "data/test",           
    "gt_volume_csv": "data/gt_volume.csv",  

    # ---- where results are written ----
    "checkpoint_dir": "checkpoints",    
    "segment_out_dir": "saved_segments",  
    "save_format": "ply",               

    # ---- ADVANCED FEATURES APPLIED ----
    "num_points": 8192,                 # Increased points per chunk
    "chunks_per_cloud": 4,              
    "val_ratio": 0.15,
    "test_ratio": 0.15,

    # ---- segmentation model training ----
    "seg_epochs": 200,
    "seg_batch_size": 4,                # Reduced for VRAM safety
    "seg_lr": 5e-4,                     # Optimized learning rate
    "grad_accum": 8,                    # Gradient Accumulation
    "amp": True,                        # Automatic Mixed Precision
    "seg_patience": 100,                 

    # ---- regression model training ----
    "reg_epochs": 60,
    "reg_batch_size": 8,
    "reg_points": 2048,                 
    "reg_lr": 1e-3,
    "reg_patience": 100,

    # ---- shared model settings ----
    "num_classes": 2,                   
    "target_class": 1,                  
    "k_neighbors": 16,                  
    "normal_k": 16,                     

    "resume": True,
    "show_windows": True,               
    "max_visualize": 50,                 
    "seed": 42,
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
os.makedirs(CONFIG["segment_out_dir"], exist_ok=True)
SEG_CKPT = os.path.join(CONFIG["checkpoint_dir"], "ptv2_segmentation.pth")
REG_CKPT = os.path.join(CONFIG["checkpoint_dir"], "ptv2_regression.pth")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(CONFIG["seed"])


# ============================================================
# 2. Load Helpers & Feature Preparation
# ============================================================
def load_points_and_labels(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".las", ".laz"):
        las = laspy.read(path)
        pts = np.column_stack((np.asarray(las.x), np.asarray(las.y),
                               np.asarray(las.z))).astype(np.float64)
        labels = None
        if "classification" in las.point_format.dimension_names:
            labels = np.asarray(las.classification, dtype=np.int64)
        return pts, labels
    for kwargs in (dict(), dict(delimiter=","), dict(skiprows=1),
                   dict(delimiter=",", skiprows=1)):
        try:
            arr = np.loadtxt(path, **kwargs)
            break
        except ValueError:
            arr = None
    if arr is None:
        arr = np.genfromtxt(path, delimiter=",", skip_header=1)
    arr = np.asarray(arr, dtype=np.float64)
    labels = arr[:, 3].astype(np.int64) if arr.shape[1] >= 4 else None
    return arr[:, :3], labels

def estimate_normals(points, k):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(k))
    pcd.orient_normals_to_align_with_direction([0.0, 0.0, 1.0])
    return np.asarray(pcd.normals, dtype=np.float32)

def make_features(points, normal_k):
    center = points.mean(axis=0, keepdims=True)
    scale = max(np.linalg.norm(points - center, axis=1).max(), 1e-9)
    norm_xyz = ((points - center) / scale).astype(np.float32)
    z = points[:, 2]
    height = ((z - z.min()) / max(z.max() - z.min(), 1e-6)).astype(np.float32)
    normals = estimate_normals(points, normal_k)
    return np.column_stack([norm_xyz, height[:, None], normals]).astype(np.float32)

def list_files(folder):
    files = []
    for ext in (".las", ".laz", ".xyz", ".pts", ".txt"):
        files += glob.glob(os.path.join(folder, "*" + ext))
    return sorted(files)

def load_gt_volumes(csv_path):
    volumes = {}
    if not os.path.exists(csv_path):
        print("WARNING: gt_volume.csv not found -> regression cannot be trained.")
        return volumes
    df = pd.read_csv(csv_path)
    filename_col, volume_col = None, None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            volume_col = col
        else:
            filename_col = col
    if filename_col is None:
        filename_col = df.columns[0]
    if volume_col is None:
        volume_col = df.columns[1]
    for _, row in df.iterrows():
        name = os.path.splitext(os.path.basename(str(row[filename_col])))[0]
        volumes[name] = float(row[volume_col])
    return volumes

all_train_files = list_files(CONFIG["train_dir"])
assert all_train_files, f"No files found in {CONFIG['train_dir']}"

rng = np.random.RandomState(CONFIG["seed"])
order = rng.permutation(len(all_train_files))
n_val = max(1, int(len(all_train_files) * CONFIG["val_ratio"]))
n_test = max(1, int(len(all_train_files) * CONFIG["test_ratio"]))

VAL_FILES   = [all_train_files[i] for i in order[:n_val]]
TEST_FILES  = [all_train_files[i] for i in order[n_val:n_val + n_test]]
TRAIN_FILES = [all_train_files[i] for i in order[n_val + n_test:]]
INFER_FILES = list_files(CONFIG["test_dir"])

print(f"train files      : {len(TRAIN_FILES)}")
print(f"validation files : {len(VAL_FILES)}")
print(f"held-out test    : {len(TEST_FILES)}")
print(f"inference files  : {len(INFER_FILES)}")

GT_VOLUMES = load_gt_volumes("data/gt_volume.csv")
print(f"ground-truth volumes loaded: {len(GT_VOLUMES)}")

FEATURE_CACHE = {}

def get_file_data(path):
    if path in FEATURE_CACHE:
        return FEATURE_CACHE[path]
    points, labels = load_points_and_labels(path)
    if labels is None:
        labels = np.zeros(len(points), dtype=np.int64)
    labels = np.clip(labels, 0, CONFIG["num_classes"] - 1).astype(np.int64)
    features = make_features(points, CONFIG["normal_k"])
    data = {"points": points, "features": features, "labels": labels}
    FEATURE_CACHE[path] = data
    return data


# ============================================================
# 3. Dataset Classes (With Advanced Augmentation)
# ============================================================
class SegmentationDataset(Dataset):
    def __init__(self, files, augment=True):
        self.files = list(files)
        self.augment = augment
        self.chunks_per_cloud = CONFIG["chunks_per_cloud"]
        self.num_points = CONFIG["num_points"]

    def __len__(self):
        return len(self.files) * self.chunks_per_cloud

    def __getitem__(self, index):
        file_path = self.files[index // self.chunks_per_cloud]
        data = get_file_data(file_path)
        features = data["features"]
        labels = data["labels"]
        n = len(features)

        if n <= self.num_points:
            idx = np.random.choice(n, self.num_points, replace=True)
        else:
            seed_point = np.random.randint(n)
            dist2 = ((features[:, :3] - features[seed_point, :3]) ** 2).sum(1)
            idx = np.argpartition(dist2, self.num_points - 1)[:self.num_points]

        x = features[idx].copy()
        y = labels[idx].copy()

        # Advanced Data Augmentation: Rotation + Scaling + Noise
        if self.augment:
            angle = np.random.uniform(0, 2 * np.pi)
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            rot = np.array([[cos_a, -sin_a, 0],
                            [sin_a,  cos_a, 0],
                            [0,      0,     1]], dtype=np.float32)
            
            scale = np.random.uniform(0.9, 1.1)
            noise = np.random.normal(0, 0.002, (len(x), 3)).astype(np.float32)
            
            x[:, 0:3] = (x[:, 0:3] @ rot.T) * scale + noise
            x[:, 4:7] = x[:, 4:7] @ rot.T 

        return torch.from_numpy(x.T), torch.from_numpy(y)

class RegressionDataset(Dataset):
    def __init__(self, files):
        self.num_points = CONFIG["reg_points"]
        self.samples = []
        for path in files:
            name = os.path.splitext(os.path.basename(path))[0]
            if name not in GT_VOLUMES:
                continue
            data = get_file_data(path)
            has_target = np.any(data["labels"] == CONFIG["target_class"])
            if has_target:
                self.samples.append((path, GT_VOLUMES[name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        file_path, volume = self.samples[index]
        data = get_file_data(file_path)
        mask = data["labels"] == CONFIG["target_class"]
        target_features = data["features"][mask]
        if len(target_features) < 8:
            target_features = data["features"]

        replace = len(target_features) < self.num_points
        idx = np.random.choice(len(target_features), self.num_points, replace=replace)
        x = target_features[idx].copy()

        y = np.log1p(volume).astype(np.float32)
        return torch.from_numpy(x.T), torch.tensor(y, dtype=torch.float32)


# ============================================================
# 4. Model Classes (Point Transformer V2)
# ============================================================
IN_CHANNELS = 7

class GroupedVectorAttention(nn.Module):
    def __init__(self, channels, groups=6, k=16):
        super().__init__()
        assert channels % groups == 0
        self.k = k
        self.groups = groups
        self.group_channels = channels // groups
        self.to_query = nn.Linear(channels, channels)
        self.to_key   = nn.Linear(channels, channels)
        self.to_value = nn.Linear(channels, channels)
        self.pos_multiplier = nn.Sequential(nn.Linear(3, channels), nn.ReLU(),
                                            nn.Linear(channels, channels))
        self.pos_bias = nn.Sequential(nn.Linear(3, channels), nn.ReLU(),
                                      nn.Linear(channels, channels))
        self.weight_net = nn.Sequential(nn.Linear(channels, channels), nn.ReLU(),
                                        nn.Linear(channels, groups))

    def forward(self, x, pos, batch):
        edges = tc_knn(pos, pos, self.k, batch, batch)
        center, neighbour = edges[0], edges[1]
        rel_pos = pos[center] - pos[neighbour]

        bias = self.pos_bias(rel_pos)
        relation = (self.to_query(x)[center] - self.to_key(x)[neighbour]) \
            * self.pos_multiplier(rel_pos) + bias
        weights = scatter_softmax(self.weight_net(relation), center, dim=0)

        values = (self.to_value(x)[neighbour] + bias).view(-1, self.groups,
                                                           self.group_channels)
        out = scatter_add(values * weights.unsqueeze(-1), center, dim=0,
                          dim_size=x.size(0))
        return out.view(-1, self.groups * self.group_channels)

class AttentionBlock(nn.Module):
    def __init__(self, channels, groups=6, k=16):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.attention = GroupedVectorAttention(channels, groups, k)
        self.mlp = nn.Sequential(nn.Linear(channels, channels * 2), nn.ReLU(),
                                 nn.Linear(channels * 2, channels))

    def forward(self, x, pos, batch):
        x = x + self.attention(self.norm1(x), pos, batch)
        x = x + self.mlp(self.norm2(x))
        return x

class GridPooling(nn.Module):
    def __init__(self, in_ch, out_ch, grid_size):
        super().__init__()
        self.grid_size = grid_size
        self.project = nn.Sequential(nn.Linear(in_ch, out_ch), nn.ReLU())

    def forward(self, x, pos, batch):
        voxel = torch.floor(pos / self.grid_size).long()
        key = torch.cat([batch.unsqueeze(1), voxel], dim=1)
        _, cluster = torch.unique(key, dim=0, return_inverse=True)
        pooled_x, _ = scatter_max(self.project(x), cluster, dim=0)
        pooled_pos = scatter_mean(pos, cluster, dim=0)
        pooled_batch = scatter_max(batch, cluster, dim=0)[0]
        return pooled_x, pooled_pos, pooled_batch, cluster

class PTv2Segmentation(nn.Module):
    def __init__(self, num_classes, k=16, dims=(48, 96, 192), groups=6):
        super().__init__()
        d0, d1, d2 = dims
        self.embed = nn.Sequential(nn.Linear(IN_CHANNELS, d0), nn.ReLU(),
                                   nn.Linear(d0, d0))
        self.enc1 = AttentionBlock(d0, groups, k)
        self.pool1 = GridPooling(d0, d1, grid_size=0.08)
        self.enc2 = AttentionBlock(d1, groups, k)
        self.pool2 = GridPooling(d1, d2, grid_size=0.16)
        self.enc3 = AttentionBlock(d2, groups, k)
        
        self.up2 = nn.Sequential(nn.Linear(d2 + d1, d1), nn.ReLU())
        self.dec2 = AttentionBlock(d1, groups, k)
        self.up1 = nn.Sequential(nn.Linear(d1 + d0, d0), nn.ReLU())
        self.dec1 = AttentionBlock(d0, groups, k)
        self.head = nn.Sequential(nn.LayerNorm(d0), nn.Linear(d0, 128), nn.ReLU(),
                                  nn.Dropout(0.4), nn.Linear(128, num_classes))

    def forward(self, x):
        batch_size, channels, num_points = x.shape
        flat = x.permute(0, 2, 1).reshape(-1, channels).contiguous()
        pos = flat[:, :3].contiguous()                 
        batch = torch.arange(batch_size, device=x.device).repeat_interleave(num_points)

        h0 = self.enc1(self.embed(flat), pos, batch)
        h1, pos1, batch1, cluster1 = self.pool1(h0, pos, batch)
        h1 = self.enc2(h1, pos1, batch1)
        h2, pos2, batch2, cluster2 = self.pool2(h1, pos1, batch1)
        h2 = self.enc3(h2, pos2, batch2)

        up1 = self.dec2(self.up2(torch.cat([h1, h2[cluster2]], dim=1)), pos1, batch1)
        up0 = self.dec1(self.up1(torch.cat([h0, up1[cluster1]], dim=1)), pos, batch)

        logits = self.head(up0)                        
        return logits.view(batch_size, num_points, -1).permute(0, 2, 1)

class PTv2Regression(nn.Module):
    def __init__(self, k=16, dims=(48, 96, 192), groups=6):
        super().__init__()
        d0, d1, d2 = dims
        self.embed = nn.Sequential(nn.Linear(IN_CHANNELS, d0), nn.ReLU(),
                                   nn.Linear(d0, d0))
        self.enc1 = AttentionBlock(d0, groups, k)
        self.pool1 = GridPooling(d0, d1, grid_size=0.08)
        self.enc2 = AttentionBlock(d1, groups, k)
        self.pool2 = GridPooling(d1, d2, grid_size=0.16)
        self.enc3 = AttentionBlock(d2, groups, k)
        self.head = nn.Sequential(nn.Linear(d2 * 2, 128), nn.ReLU(),
                                  nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, x):
        batch_size, channels, num_points = x.shape
        flat = x.permute(0, 2, 1).reshape(-1, channels).contiguous()
        pos = flat[:, :3].contiguous()
        batch = torch.arange(batch_size, device=x.device).repeat_interleave(num_points)

        h0 = self.enc1(self.embed(flat), pos, batch)
        h1, pos1, batch1, _ = self.pool1(h0, pos, batch)
        h1 = self.enc2(h1, pos1, batch1)
        h2, pos2, batch2, _ = self.pool2(h1, pos1, batch1)
        h2 = self.enc3(h2, pos2, batch2)

        pooled_max, _ = scatter_max(h2, batch2, dim=0, dim_size=batch_size)
        pooled_mean = scatter_mean(h2, batch2, dim=0, dim_size=batch_size)
        summary = torch.cat([pooled_max, pooled_mean], dim=1)
        return self.head(summary).squeeze(-1)          


# ============================================================
# 5. Metrics
# ============================================================
def compute_confusion(true_labels, pred_labels, num_classes):
    k = num_classes
    return np.bincount(true_labels * k + pred_labels,
                       minlength=k * k).reshape(k, k)

def confusion_to_scores(conf):
    conf = conf.astype(np.float64)
    accuracy = np.diag(conf).sum() / max(conf.sum(), 1)
    ious = []
    for c in range(conf.shape[0]):
        tp = conf[c, c]
        fp = conf[:, c].sum() - tp
        fn = conf[c, :].sum() - tp
        iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else float("nan")
        ious.append(iou)
    mean_iou = float(np.nanmean(ious))
    return {"accuracy": float(accuracy), "iou_per_class": ious, "miou": mean_iou}


# ============================================================
# 6. Training Functions (With AMP & Accumulation)
# ============================================================
def train_segmentation():
    """Train PTv2Segmentation with AMP and Gradient Accumulation."""
    model = PTv2Segmentation(CONFIG["num_classes"], k=CONFIG["k_neighbors"]).to(DEVICE)

    if CONFIG["resume"] and os.path.exists(SEG_CKPT):
        print(f"[resume] found {SEG_CKPT} -> loading, skipping segmentation training")
        model.load_state_dict(torch.load(SEG_CKPT, map_location=DEVICE)["model_state"])
        return model

    train_loader = DataLoader(SegmentationDataset(TRAIN_FILES, augment=True),
                              batch_size=CONFIG["seg_batch_size"], shuffle=True,
                              drop_last=True)
    val_loader = DataLoader(SegmentationDataset(VAL_FILES, augment=False),
                            batch_size=CONFIG["seg_batch_size"], shuffle=False)

    counts = np.zeros(CONFIG["num_classes"])
    for f in TRAIN_FILES[:30]:
        counts += np.bincount(get_file_data(f)["labels"],
                              minlength=CONFIG["num_classes"])
    weights = 1.0 / np.clip(counts / counts.sum(), 1e-6, None)
    weights = torch.tensor(weights / weights.sum() * CONFIG["num_classes"],
                           dtype=torch.float32, device=DEVICE)

    loss_fn = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["seg_lr"], weight_decay=1e-4)
    
    accum = max(1, CONFIG.get("grad_accum", 1))
    use_amp = CONFIG.get("amp", False) and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)

    best_miou = -1.0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, CONFIG["seg_epochs"] + 1):
        model.train()
        optimizer.zero_grad() 
        
        for step, (x, y) in enumerate(train_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)
            
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                logits = model(x)
                loss = loss_fn(logits, y) / accum 
                
            scaler.scale(loss).backward()
            
            if (step + 1) % accum == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        model.eval()
        conf = np.zeros((CONFIG["num_classes"],) * 2, dtype=np.int64)
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                    preds = model(x).argmax(dim=1)
                conf += compute_confusion(y.cpu().numpy().ravel(),
                                          preds.cpu().numpy().ravel(),
                                          CONFIG["num_classes"])
        scores = confusion_to_scores(conf)
        note = ""
        if scores["miou"] > best_miou + 1e-4:
            best_miou = scores["miou"]
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
            note = "  <- best so far"
        else:
            epochs_without_improvement += 1
        print(f"epoch {epoch:3d} | val accuracy {scores['accuracy']:.4f} | "
              f"val mIoU {scores['miou']:.4f}{note}")

        if epochs_without_improvement >= CONFIG["seg_patience"]:
            print(f"early stop: no improvement for {CONFIG['seg_patience']} epochs")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save({"model_state": model.state_dict(), "best_miou": best_miou}, SEG_CKPT)
    print(f"saved best segmentation model -> {SEG_CKPT} (val mIoU {best_miou:.4f})")
    return model


def train_regression():
    model = PTv2Regression(k=CONFIG["k_neighbors"]).to(DEVICE)

    if CONFIG["resume"] and os.path.exists(REG_CKPT):
        print(f"[resume] found {REG_CKPT} -> loading, skipping regression training")
        model.load_state_dict(torch.load(REG_CKPT, map_location=DEVICE)["model_state"])
        return model

    full_dataset = RegressionDataset(TRAIN_FILES + VAL_FILES)
    if len(full_dataset) < 4:
        print("Not enough files with both target points and GT volume -> "
              "skipping regression training.")
        return None

    n_val = max(1, int(0.2 * len(full_dataset)))
    val_subset = torch.utils.data.Subset(full_dataset, list(range(n_val)))
    train_subset = torch.utils.data.Subset(full_dataset, list(range(n_val, len(full_dataset))))

    train_loader = DataLoader(train_subset, batch_size=CONFIG["reg_batch_size"],
                              shuffle=True, drop_last=len(train_subset) > CONFIG["reg_batch_size"])
    val_loader = DataLoader(val_subset, batch_size=CONFIG["reg_batch_size"])

    loss_fn = nn.SmoothL1Loss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["reg_lr"], weight_decay=1e-4)

    best_mae = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, CONFIG["reg_epochs"] + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()

        model.eval()
        errors = []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(DEVICE)
                pred_volume = np.expm1(model(x).cpu().numpy())
                true_volume = np.expm1(y.numpy())
                errors.extend(np.abs(pred_volume - true_volume).tolist())
        mae = float(np.mean(errors)) if errors else float("inf")
        note = ""
        if mae < best_mae - 1e-6:
            best_mae = mae
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
            note = "  <- best so far"
        else:
            epochs_without_improvement += 1
        print(f"epoch {epoch:3d} | val volume MAE {mae:.4f}{note}")

        if epochs_without_improvement >= CONFIG["reg_patience"]:
            print(f"early stop: no improvement for {CONFIG['reg_patience']} epochs")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save({"model_state": model.state_dict(), "best_mae": best_mae}, REG_CKPT)
    print(f"saved best regression model -> {REG_CKPT} (val MAE {best_mae:.4f})")
    return model


# ============================================================
# 7. Prediction & Visualization Helpers
# ============================================================
@torch.no_grad()
def segment_whole_cloud(seg_model, features):
    seg_model.eval()
    num_points = CONFIG["num_points"]
    batch_size = CONFIG["seg_batch_size"]
    n = len(features)

    order = np.random.RandomState(0).permutation(n)
    pad = (num_points - n % num_points) % num_points
    if pad:
        order = np.concatenate([order, order[:pad]])
    chunks = order.reshape(-1, num_points)

    predictions = np.zeros(n, dtype=np.int64)
    for start in range(0, len(chunks), batch_size):
        chunk_ids = chunks[start:start + batch_size]
        x = torch.from_numpy(features[chunk_ids].transpose(0, 2, 1)).to(DEVICE)
        preds = seg_model(x).argmax(dim=1).cpu().numpy()
        predictions[chunk_ids.ravel()] = preds.ravel()
    return predictions

@torch.no_grad()
def estimate_volume(reg_model, target_features):
    if reg_model is None or len(target_features) < 8:
        return float("nan")
    reg_model.eval()
    num_points = CONFIG["reg_points"]
    replace = len(target_features) < num_points
    idx = np.random.RandomState(0).choice(len(target_features), num_points, replace=replace)
    x = torch.from_numpy(target_features[idx].T).unsqueeze(0).to(DEVICE)
    pred_log = reg_model(x).item()
    return max(float(np.expm1(pred_log)), 0.0)

def save_segment(points, filename):
    out_dir = CONFIG["segment_out_dir"]
    if CONFIG["save_format"] == "las":
        header = laspy.LasHeader(point_format=3, version="1.2")
        las = laspy.LasData(header)
        las.x, las.y, las.z = points[:, 0], points[:, 1], points[:, 2]
        path = os.path.join(out_dir, filename + "_target.las")
        las.write(path)
    else:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
        pcd.paint_uniform_color([0.0, 0.8, 0.0])       
        path = os.path.join(out_dir, filename + "_target.ply")
        o3d.io.write_point_cloud(path, pcd)
    return path

def show_segment_window(points_all, predictions, title):
    """Open an Open3D window: showing ONLY the segmented target points."""
    if not CONFIG["show_windows"]:
        return
    
    is_target = predictions == CONFIG["target_class"]
    target_points = points_all[is_target]
    
    if len(target_points) == 0:
        return

    colors = np.zeros((len(target_points), 3))
    colors[:] = [0.0, 0.8, 0.0]                # green = target
    
    pcd = o3d.geometry.PointCloud()
    centered_points = target_points.astype(np.float64) - np.mean(target_points.astype(np.float64), axis=0)
    pcd.points = o3d.utility.Vector3dVector(centered_points)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    o3d.visualization.draw_geometries([pcd], window_name=title, width=1280, height=800)


# ============================================================
# 8. Testing Phase
# ============================================================
def test(seg_model):
    print("\n===== TESTING (held-out labelled files) =====")
    total_conf = np.zeros((CONFIG["num_classes"],) * 2, dtype=np.int64)
    shown = 0

    for path in TEST_FILES:
        name = os.path.splitext(os.path.basename(path))[0]
        data = get_file_data(path)
        predictions = segment_whole_cloud(seg_model, data["features"])

        total_conf += compute_confusion(data["labels"], predictions, CONFIG["num_classes"])

        target_points = data["points"][predictions == CONFIG["target_class"]]
        if len(target_points) > 0:
            saved = save_segment(target_points, name)
            print(f"{name}: {len(target_points)} target points -> saved {saved}")
        else:
            print(f"{name}: no target points predicted")

        if shown < CONFIG["max_visualize"]:
            show_segment_window(data["points"], predictions, title=f"TEST | {name} | green = target")
            shown += 1

    scores = confusion_to_scores(total_conf)
    print(f"\nOverall test accuracy : {scores['accuracy']:.4f}")
    print(f"Overall test mIoU     : {scores['miou']:.4f}")
    for c in range(CONFIG["num_classes"]):
        print(f"  class {c} IoU: {scores['iou_per_class'][c]:.4f}")


# ============================================================
# 9. Inference Phase
# ============================================================
def inference(seg_model, reg_model):
    print("\n===== INFERENCE (data/test, unlabelled) =====")
    if not INFER_FILES:
        print("No files in data/test -> nothing to do.")
        return

    shown = 0
    for path in INFER_FILES:
        name = os.path.splitext(os.path.basename(path))[0]
        data = get_file_data(path)
        predictions = segment_whole_cloud(seg_model, data["features"])

        target_mask = predictions == CONFIG["target_class"]
        target_points = data["points"][target_mask]
        target_features = data["features"][target_mask]

        volume = estimate_volume(reg_model, target_features)

        if len(target_points) > 0:
            saved = save_segment(target_points, name)
            print(f"{name}: {len(target_points)} target points | "
                  f"estimated volume = {volume:.4f} -> saved {saved}")
        else:
            print(f"{name}: no target points predicted | volume = n/a")

        if shown < CONFIG["max_visualize"]:
            show_segment_window(data["points"], predictions,
                                title=f"INFERENCE | {name} | volume = {volume:.4f}")
            shown += 1

# ============================================================
# 10. Run Everything
# ============================================================
seg_model = train_segmentation()
reg_model = train_regression()

test(seg_model)
inference(seg_model, reg_model)

print("\nAll done.")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
PyTorch version : 2.5.1+cu121
Running on      : cuda
train files      : 33
validation files : 6
held-out test    : 6
inference files  : 18
ground-truth volumes loaded: 63
epoch   1 | val accuracy 0.4289 | val mIoU 0.2144  <- best so far
epoch   2 | val accuracy 0.4924 | val mIoU 0.3266  <- best so far
epoch   3 | val accuracy 0.5121 | val mIoU 0.2560
epoch   4 | val accuracy 0.4400 | val mIoU 0.2200
epoch   5 | val accuracy 0.4982 | val mIoU 0.2557
epoch   6 | val accuracy 0.5170 | val mIoU 0.3132
epoch   7 | val accuracy 0.7956 | val mIoU 0.6261  <- best so far
epoch   8 | val accuracy 0.6486 | val mIoU 0.4304
epoch   9 | val accuracy 0.5368 | val mIoU 0.3470
epoch  10 | val accuracy 0.6075 | val mIoU 0.3978
epoch  11 | val accuracy 0.5950 | val mIoU 0.4141
epoch  12 | val accuracy 0.6797 | val mIoU 0.5077
epoch  13 | 

KeyboardInterrupt: 